In [ ]:
import sys; sys.path.insert(0, "..")
import pandas as pd, torch
from src.data import PROCESSED
from src.dl_dataset import load_cache
from src.train_dl import run_cv_experiment

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
folds_df = pd.read_csv(PROCESSED / "folds.csv")
train_cache = load_cache(folds_df["record"].unique())

GPU: NVIDIA RTX A2000 8GB Laptop GPU


In [2]:
# Main Model — RR + EDR (3-min context) on strict patient-wise CV
oof_main, m_main = run_cv_experiment(
    mode="rr_edr", context_mins=3, split_col="patient_fold",
    run_name="cnn_bigru_rredr_3m", cache=train_cache, use_wandb=True
)

KeyboardInterrupt: 

In [ ]:
# Ablations on the exact same patient_fold split
oof_1m, m_1m = run_cv_experiment(mode="rr_edr", context_mins=1, split_col="patient_fold", run_name="cnn_bigru_rredr_1m", cache=train_cache)
oof_rr, m_rr = run_cv_experiment(mode="rr", context_mins=3, split_col="patient_fold", run_name="cnn_bigru_rr_3m",cache=train_cache)
oof_raw, m_raw = run_cv_experiment(mode="raw_ecg", context_mins=1, split_col="patient_fold", run_name="cnn_bigru_rawecg", cache=train_cache)

In [ ]:
# Leakage Audit — random minute-level CV vs patient-wise CV
_, m_main_rand = run_cv_experiment(mode="rr_edr", context_mins=1, split_col="random_fold", run_name="cnn_bigru_rredr_1m", cache=train_cache)
_, m_raw_rand = run_cv_experiment(mode="raw_ecg", context_mins=1, split_col="random_fold", run_name="cnn_bigru_rawecg", cache=train_cache)